# Topic 1: Unit Testing PySpark

> **Phase 7 → Week 15 → Topic 1**

---

## Why Test Spark Pipelines?

Data pipelines are software. Untested pipelines break silently — wrong numbers go to dashboards for weeks before anyone notices. Testing gives you:

```
Confidence to refactor:  change partitioning strategy → tests catch regressions
Documentation:           tests show WHAT the transformation is supposed to do
Fast feedback:           test locally in 5 seconds vs 30-minute EMR job
```

---

## Interview Cheat Sheet

**Q: How do you unit test a PySpark transformation?**
> Use pytest with a session-scoped `SparkSession` fixture. Create small test DataFrames with `spark.createDataFrame()`. Call the transformation function. Assert on the result using `.collect()` or `.toPandas()`. Keep test data minimal — 5-10 rows covering all edge cases. Never write to external systems in unit tests (no S3, no database) — test the transformation logic in isolation.

**Q: What is a pytest fixture and how do you use it for SparkSession?**
> A fixture is a function decorated with `@pytest.fixture` that provides setup/teardown for tests. For SparkSession, use `scope='session'` so one SparkSession is created for the entire test run (not one per test). The fixture value is injected into tests via parameter name matching. Use `conftest.py` so the fixture is available to all test files without importing.

**Q: What is the difference between unit, integration, and end-to-end tests for Spark?**
> Unit: test one transformation function with in-memory DataFrames, no external I/O. Integration: test a multi-step pipeline reading from/writing to local disk or a test S3 bucket. End-to-end: run the full pipeline in a staging environment with real data. Unit tests: run in seconds, run on every commit. Integration: run in minutes, run on PR. E2E: run in hours, run before production deploy.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week15 - Unit Testing") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Unit Testing PySpark — ready")

## Part 1: Structuring Testable Code

In [ ]:
# ── The transformation module (silver_transform.py) ──────────
# Good: pure functions that take a DataFrame and return a DataFrame
# Testable without any I/O, without S3, without a real cluster

from pyspark.sql import DataFrame

def validate_orders(df: DataFrame) -> DataFrame:
    """Remove invalid orders: null customer, non-positive amount."""
    return df.filter(
        F.col("customer_id").isNotNull() &
        (F.col("amount") > 0)
    )

def deduplicate_orders(df: DataFrame, key_col: str = "order_id") -> DataFrame:
    """Keep first occurrence of each order_id (by event_time)."""
    from pyspark.sql import Window
    w = Window.partitionBy(key_col).orderBy("event_time")
    return df \
        .withColumn("_rn", F.row_number().over(w)) \
        .filter(F.col("_rn") == 1) \
        .drop("_rn")

def enrich_with_tier(df: DataFrame) -> DataFrame:
    """Add customer tier based on amount."""
    return df.withColumn(
        "tier",
        F.when(F.col("amount") >= 500, "gold")
         .when(F.col("amount") >= 200, "silver")
         .otherwise("bronze")
    )

def transform_silver(df: DataFrame) -> DataFrame:
    """Full Silver transformation pipeline."""
    return (
        df
        .pipe(validate_orders)
        .pipe(deduplicate_orders)
        .pipe(enrich_with_tier)
    )

print("Transformation functions defined")

## Part 2: pytest Test File (conftest.py + test_silver.py)

In [ ]:
print("""
conftest.py — shared SparkSession fixture:
══════════════════════════════════════════════════════════════

# tests/conftest.py
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope='session')
def spark():
    """One SparkSession shared across ALL tests in the session."""
    session = (
        SparkSession.builder
        .appName('test')
        .master('local[2]')
        .config('spark.sql.shuffle.partitions', '2')
        .config('spark.default.parallelism', '2')
        .getOrCreate()
    )
    session.sparkContext.setLogLevel('ERROR')
    yield session
    session.stop()


test_silver_transform.py — unit tests:
══════════════════════════════════════════════════════════════

# tests/test_silver_transform.py
import pytest
from pyspark.sql import functions as F
from silver_transform import validate_orders, deduplicate_orders, enrich_with_tier

class TestValidateOrders:
    def test_removes_null_customer(self, spark):
        df = spark.createDataFrame([
            ('O1', None, 100.0),
            ('O2', 'C1',  50.0),
        ], ['order_id', 'customer_id', 'amount'])

        result = validate_orders(df)

        assert result.count() == 1
        assert result.collect()[0]['order_id'] == 'O2'

    def test_removes_non_positive_amount(self, spark):
        df = spark.createDataFrame([
            ('O1', 'C1', -10.0),
            ('O2', 'C1',   0.0),
            ('O3', 'C1',  50.0),
        ], ['order_id', 'customer_id', 'amount'])

        result = validate_orders(df)

        assert result.count() == 1
        assert result.collect()[0]['order_id'] == 'O3'

    def test_passes_valid_rows(self, spark):
        df = spark.createDataFrame([
            ('O1', 'C1', 100.0),
            ('O2', 'C2', 200.0),
        ], ['order_id', 'customer_id', 'amount'])

        assert validate_orders(df).count() == 2

    def test_empty_dataframe(self, spark):
        from pyspark.sql.types import StructType, StructField, StringType, DoubleType
        schema = StructType([
            StructField('order_id', StringType()),
            StructField('customer_id', StringType()),
            StructField('amount', DoubleType()),
        ])
        df = spark.createDataFrame([], schema)
        assert validate_orders(df).count() == 0


class TestEnrichWithTier:
    @pytest.mark.parametrize('amount,expected_tier', [
        (500.0, 'gold'),
        (600.0, 'gold'),
        (200.0, 'silver'),
        (499.9, 'silver'),
        (199.9, 'bronze'),
        (1.0,   'bronze'),
    ])
    def test_tier_boundaries(self, spark, amount, expected_tier):
        df = spark.createDataFrame([(amount,)], ['amount'])
        result = enrich_with_tier(df).collect()[0]
        assert result['tier'] == expected_tier, f"{amount} → expected {expected_tier}, got {result['tier']}"
══════════════════════════════════════════════════════════════
""")

## Part 3: Running the Tests In-Notebook

In [ ]:
# Run tests directly in-notebook (no pytest needed here)
import traceback

def assert_equal(actual, expected, msg=""):
    assert actual == expected, f"{msg}\n  Expected: {expected}\n  Actual:   {actual}"

def run_test(name, fn):
    try:
        fn()
        print(f"  ✅ {name}")
    except AssertionError as e:
        print(f"  ❌ {name}: {e}")
    except Exception as e:
        print(f"  💥 {name}: {type(e).__name__}: {e}")

print("=== test_validate_orders ===")

run_test("removes null customer", lambda: assert_equal(
    validate_orders(spark.createDataFrame(
        [("O1", None, 100.0), ("O2", "C1", 50.0)],
        ["order_id", "customer_id", "amount"]
    )).count(), 1
))

run_test("removes zero and negative amounts", lambda: assert_equal(
    validate_orders(spark.createDataFrame(
        [("O1", "C1", -10.0), ("O2", "C1", 0.0), ("O3", "C1", 50.0)],
        ["order_id", "customer_id", "amount"]
    )).count(), 1
))

run_test("passes all valid rows", lambda: assert_equal(
    validate_orders(spark.createDataFrame(
        [("O1", "C1", 100.0), ("O2", "C2", 200.0)],
        ["order_id", "customer_id", "amount"]
    )).count(), 2
))

print("\n=== test_enrich_with_tier ===")

tier_cases = [
    (500.0, "gold"), (600.0, "gold"),
    (200.0, "silver"), (499.9, "silver"),
    (199.9, "bronze"), (1.0, "bronze"),
]
for amount, expected in tier_cases:
    run_test(
        f"amount={amount} → tier={expected}",
        lambda a=amount, e=expected: assert_equal(
            enrich_with_tier(spark.createDataFrame([(a,)], ["amount"])).collect()[0]["tier"], e
        )
    )

print("\n=== test_deduplicate_orders ===")

run_test("keeps first by event_time", lambda: assert_equal(
    deduplicate_orders(
        spark.createDataFrame([
            ("O1", "C1", 100.0, "2024-01-15 10:00"),
            ("O1", "C1", 100.0, "2024-01-15 10:05"),  # duplicate, later → dropped
            ("O2", "C2", 200.0, "2024-01-15 10:01"),
        ], ["order_id", "customer_id", "amount", "event_time"])
    ).count(), 2
))

print("\nAll tests done.")

## Part 4: Schema Assertion Helper

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType
from typing import List

def assert_schema(df: DataFrame, expected_cols: List[str]):
    """Assert that DataFrame has exactly the expected columns (order-insensitive)."""
    actual = sorted(df.columns)
    expected = sorted(expected_cols)
    assert actual == expected, f"Schema mismatch.\nExpected: {expected}\nActual:   {actual}"

def assert_no_nulls(df: DataFrame, col: str):
    """Assert that a column has no null values."""
    null_count = df.filter(F.col(col).isNull()).count()
    assert null_count == 0, f"Column '{col}' has {null_count} null(s)"

def assert_unique(df: DataFrame, col: str):
    """Assert that a column has no duplicate values."""
    total   = df.count()
    distinct = df.select(col).distinct().count()
    assert total == distinct, f"Column '{col}' has {total - distinct} duplicate(s)"

def assert_row_count(df: DataFrame, expected: int):
    actual = df.count()
    assert actual == expected, f"Row count: expected {expected}, got {actual}"

# Demo
result = transform_silver(
    spark.createDataFrame([
        ("O1", "C1", 250.0, "2024-01-15 10:00"),
        ("O2", None,  50.0, "2024-01-15 10:01"),  # filtered
        ("O1", "C1", 250.0, "2024-01-15 10:05"),  # deduped
        ("O3", "C2", 600.0, "2024-01-15 10:02"),
    ], ["order_id", "customer_id", "amount", "event_time"])
)

print("=== Assertion helpers ===")
run_test("schema has correct columns", lambda: assert_schema(
    result, ["order_id", "customer_id", "amount", "event_time", "tier"]
))
run_test("no null customer_id",  lambda: assert_no_nulls(result, "customer_id"))
run_test("order_id unique",      lambda: assert_unique(result, "order_id"))
run_test("2 rows survived",      lambda: assert_row_count(result, 2))

print("\nFinal result:")
result.show()

## Part 5: pytest.ini & Running Tests

In [ ]:
print("""
Project structure for testable Spark code:
══════════════════════════════════════════════════════════════

etl_project/
├── src/
│   ├── __init__.py
│   ├── bronze_ingest.py
│   ├── silver_transform.py    ← pure transformation functions
│   └── gold_build.py
├── tests/
│   ├── conftest.py            ← shared SparkSession fixture
│   ├── test_bronze_ingest.py
│   ├── test_silver_transform.py
│   └── test_gold_build.py
├── pytest.ini
└── requirements-dev.txt

pytest.ini:
  [pytest]
  testpaths = tests
  python_files = test_*.py
  python_classes = Test*
  python_functions = test_*
  addopts = -v --tb=short
  # Speed up by not creating new JVM per test file:
  # SparkSession is session-scoped in conftest.py

requirements-dev.txt:
  pyspark==3.5.0
  pytest==8.0.0
  pytest-cov==5.0.0       # coverage reporting
  delta-spark==3.0.0
  chispa==0.9.4            # DataFrame equality assertions (optional)

Run tests:
  pytest tests/ -v                    # all tests, verbose
  pytest tests/test_silver.py -v      # one file
  pytest tests/ -k 'test_tier'        # tests matching keyword
  pytest tests/ --cov=src --cov-report=html  # with coverage

chispa — DataFrame equality assertion library:
  from chispa import assert_df_equality
  assert_df_equality(result_df, expected_df, ignore_row_order=True)
  # Checks schema + data in one call, clear diffs on failure
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Write `conftest.py` with a session-scoped `spark` fixture. Then write 5 unit tests for a `compute_daily_revenue(df)` function that groups by `customer_id` and `date`, sums `amount`, and filters for `status='shipped'`. Cover: normal case, all-filtered case, multiple customers same day, empty DataFrame.
2. Write a parametrized pytest test for `enrich_with_tier` covering all boundary values (include 200.0, 199.9, 500.0, 499.9). Use `@pytest.mark.parametrize`.
3. Write `assert_schema`, `assert_no_nulls`, `assert_unique`, and `assert_value_range` helper functions. Write tests that use them against a Silver transformation output.
4. Structure a PySpark ETL project so that `silver_transform.py` contains only pure functions (DataFrame in → DataFrame out), and the main script (`main.py`) handles I/O. Explain why this separation makes testing easier.
5. What is `chispa` and how does `assert_df_equality(result, expected, ignore_row_order=True)` improve on manual `.collect()` comparisons? What does it show on failure that manual assertion doesn't?